In [1]:
import os
import random
import numpy as np
from skimage import io
from PIL import Image
import matplotlib.pyplot as plt

def _supcls_pick_binarize_smart(super_map, min_area_threshold=100):
    """
    智能地選取一個superpixel作為前景。
    1. 過濾掉面積過小的區域。
    2. 根據面積大小進行加權隨機採樣。

    Args:
        super_map (np.array): 多標籤的偽標籤圖。
        min_area_threshold (int): 被視為有效區域的最小像素面積。

    Returns:
        tuple: (binary_mask, selected_id)
    """
    # 1. 獲取所有唯一的標籤ID及其對應的像素數量(面積)
    unique_ids, counts = np.unique(super_map, return_counts=True)
    
    print("所有区域的ID和面积:")
    for uid, count in zip(unique_ids, counts):
        if uid != 0: # 忽略背景
            print(f"  ID: {uid}, 面积: {count}")

    # 2. 過濾掉背景(ID=0)和面積過小的區域
    #    創建一個布林遮罩來篩選
    valid_indices = (unique_ids != 0) & (counts >= min_area_threshold)
    
    # 獲取合格的ID和它們對應的面積
    valid_ids = unique_ids[valid_indices]
    print(valid_ids)
    valid_areas = counts[valid_indices]

    # 3. 檢查是否還有合格的區域可供選擇
    if len(valid_ids) == 0:
        print(f"警告: 過濾後 (最小面積 > {min_area_threshold} 像素)，沒有找到任何有效的超級像素。")
        # 返回一個空的遮罩和無效ID
        return np.zeros_like(super_map, dtype=np.float32), -1

    # 4. 根據面積進行加權隨機採樣
    #    random.choices 允許我們提供一個權重列表
    #    面積越大的ID，被選中的概率越高
    # weights = np.log(valid_areas + 1)
    weights = valid_areas
    random_id = random.choices(population=valid_ids, weights=weights, k=1)[0]
    
    print(f"共有 {len(valid_ids)} 個合格的超級像素。本次根據面積加權隨機選中的ID為: {random_id}")
    
    # 5. 創建二元遮罩
    binary_mask = (super_map == random_id).astype(np.float32)
    
    return binary_mask, random_id

def main():
    """
    主函式，用於設定路徑並執行檢查。
    """
    # --- 1. 在此處設定您要檢查的檔案路徑 ---
    
    # 指定您想檢查的單張原始圖像的路徑
    IMAGE_PATH = "/home/U116med/data/polyp/TrainDataset/images/586.png"
    
    # 指定與上述圖像對應的偽標籤檔案的路徑
    # 通常是 '_label.png' 結尾
    PSEUDOLABEL_PATH = '/home/U116med/wch_code/ssl_fss/pvt_kmeans/polypdata/pvt_kmeans_masks_stage1/586_mask.png'
    
    # 指定輸出的黑白遮罩圖像的儲存路徑
    OUTPUT_MASK_PATH = './check_output_mask_slic_kmeans.png'
    
    # (可選) 指定一個包含原圖、偽標籤和最終遮罩的對比圖儲存路徑
    OUTPUT_VISUALIZATION_PATH = './check_visualization_slic_kmeans.png'
    
    # --- 2. 檢查檔案是否存在 ---
    if not os.path.exists(IMAGE_PATH):
        print(f"錯誤: 找不到圖像檔案: {IMAGE_PATH}")
        return
    if not os.path.exists(PSEUDOLABEL_PATH):
        print(f"錯誤: 找不到偽標籤檔案: {PSEUDOLABEL_PATH}")
        return

    # --- 3. 執行核心邏輯 ---
    print("正在載入偽標籤圖...")
    # 使用 as_gray=True 確保以單通道讀取
    pseudolabel_map = io.imread(PSEUDOLABEL_PATH, as_gray=True)
    # 確保資料類型為整數
    pseudolabel_map = pseudolabel_map.astype(np.int32)
    
    # 獲取最大標籤 ID
    max_label_id = pseudolabel_map.max()

    # 如果圖片全黑，直接退出
    if max_label_id < 1:
        print("這張偽標籤圖是空的，無法抽取。")
        return

    print("正在智能地抽取一個超級像素並生成二元遮罩...")
    # 可以調整 min_area_threshold 參數
    binary_mask, selected_id = _supcls_pick_binarize_smart(pseudolabel_map, min_area_threshold=300)

    # 檢查是否成功抽取
    if selected_id == -1:
        print("無法生成遮罩，程序終止。")
        return

    # --- 4. 儲存輸出的黑白遮罩 ---
    # 將 float 類型的 0.0/1.0 遮罩轉換為 uint8 類型的 0/255 黑白圖像
    mask_to_save = (binary_mask * 255).astype(np.uint8)
    io.imsave(OUTPUT_MASK_PATH, mask_to_save)
    print(f"已成功儲存隨機抽取的二元遮罩至: {OUTPUT_MASK_PATH}")

    # --- 5. (可選) 產生並儲存視覺化對比圖 ---
    try:
        original_image = io.imread(IMAGE_PATH)
        
        # 為了視覺化，為多標籤偽標籤圖上色
        # 使用 matplotlib 的 'tab20' colormap，它有很多不同的顏色
        colored_pseudolabel = plt.cm.tab20(pseudolabel_map / max(1, max_label_id))[:, :, :3] * 255
        
        # 創建一個三通道版本的黑白遮罩
        binary_mask_rgb = np.stack([mask_to_save]*3, axis=-1)
        
        # 確保所有圖像尺寸一致
        h, w, _ = original_image.shape
        colored_pseudolabel = cv2.resize(colored_pseudolabel.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        binary_mask_rgb = cv2.resize(binary_mask_rgb.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)

        # 添加標題
        font = cv2.FONT_HERSHEY_SIMPLEX
        def add_title(image, text):
            img_with_title = cv2.copyMakeBorder(image, 40, 0, 0, 0, cv2.BORDER_CONSTANT, value=[255, 255, 255])
            cv2.putText(img_with_title, text, (10, 30), font, 1, (0, 0, 0), 2, cv2.LINE_AA)
            return img_with_title
        
        # 轉換顏色通道以便 OpenCV 處理
        original_image_bgr = cv2.cvtColor(original_image, cv2.COLOR_RGB2BGR)
        colored_pseudolabel_bgr = cv2.cvtColor(colored_pseudolabel, cv2.COLOR_RGB2BGR)
        binary_mask_bgr = cv2.cvtColor(binary_mask_rgb, cv2.COLOR_RGB2BGR)

        # 添加標題
        img1 = add_title(original_image_bgr, "Original Image")
        img2 = add_title(colored_pseudolabel_bgr, "Pseudolabel Map")
        img3 = add_title(binary_mask_bgr, f"Random Mask (ID: {selected_id})")

        # 拼接圖像
        combined_image = np.concatenate((img1, img2, img3), axis=1)

        # 儲存對比圖
        cv2.imwrite(OUTPUT_VISUALIZATION_PATH, combined_image)
        print(f"已成功儲存視覺化對比圖至: {OUTPUT_VISUALIZATION_PATH}")

    except Exception as e:
        print(f"產生視覺化對比圖時發生錯誤: {e}")
        print("請確保您已安裝 opencv-python 和 matplotlib: pip install opencv-python matplotlib")


if __name__ == '__main__':
    # 確保您安裝了必要的庫
    try:
        import cv2
    except ImportError:
        print("錯誤: 找不到 OpenCV。請運行 'pip install opencv-python'")
        exit()
    main()

正在載入偽標籤圖...
正在智能地抽取一個超級像素並生成二元遮罩...
所有区域的ID和面积:
  ID: 1, 面积: 19145
  ID: 2, 面积: 20112
  ID: 3, 面积: 12239
  ID: 4, 面积: 28581
  ID: 5, 面积: 8496
  ID: 6, 面积: 7384
  ID: 7, 面积: 5352
  ID: 8, 面积: 9283
[1 2 3 4 5 6 7 8]
共有 8 個合格的超級像素。本次根據面積加權隨機選中的ID為: 4
已成功儲存隨機抽取的二元遮罩至: ./check_output_mask_slic_kmeans.png
已成功儲存視覺化對比圖至: ./check_visualization_slic_kmeans.png
